# 🏥 Insulin Model Retraining - Stabilized Configuration

## Problem Diagnosis
Previous training had critical configuration errors:
1. **Hardcoded `scale_pos_weight=50`** when actual ratio was ~3.0 → Gradient instability
2. **Missing `max_delta_step`** → Exploding updates on positive samples

## Solution
- **Dynamic ratio calculation** from actual data
- **Stabilized hyperparameters** for zero-inflated regression
- **Early stopping** with proper validation

---

## 📦 Setup

In [ ]:
# Install & Mount
!pip install polars xgboost -q

from google.colab import drive
drive.mount('/content/drive')

print("✅ Setup complete")

In [ ]:
# GPU Check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Imports
import os
import time
import polars as pl
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import gc

print(f"✅ Polars: {pl.__version__}")
print(f"✅ XGBoost: {xgb.__version__}")

## ⚙️ Configuration

In [ ]:
# ============================================================
# PATH CONFIGURATION
# ============================================================
BASE_DIR = "/content/drive/MyDrive/DiabetesKDS"
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")

# Files
TRAIN_FILE = os.path.join(DATA_DIR, "train_augmented_fixed.parquet")
VAL_FILE = os.path.join(DATA_DIR, "test_ready_for_xgboost.parquet")
MODEL_OUTPUT = os.path.join(DATA_DIR, "xgboost_final_model_v3_stabilized.json")

# Features (strictly numerical)
FEATURES = [
    'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min', 'basal',
    'age', 'weight', 'gender_encoded', 'insulin_type_encoded'
]
TARGET = 'bolus'

# Encoding Maps (for preprocessing)
GENDER_MAP = {'Male': 0, 'Female': 1, 'M': 0, 'F': 1}
INSULIN_MAP = {
    'Humalog': 3, 'Novolog': 6, 'NovoLog': 6, 'Fiasp': 2,
    'Apidra': 0, 'Regular': 8, 'Humalog (Lispro)': 4
}

print("=" * 60)
print("📂 CONFIGURATION")
print("=" * 60)
print(f"Train: {TRAIN_FILE}")
print(f"Val:   {VAL_FILE}")
print(f"Output: {MODEL_OUTPUT}")

# Validate paths
for f in [TRAIN_FILE, VAL_FILE]:
    exists = os.path.exists(f)
    print(f"{'✅' if exists else '❌'} {os.path.basename(f)}")

## 📊 Step 1: Load Data & Calculate Imbalance Ratio

In [ ]:
# ============================================================
# LOAD TRAINING DATA
# ============================================================
print("\n" + "=" * 60)
print("📊 STEP 1: DATA LOADING")
print("=" * 60)

df_train = pl.read_parquet(TRAIN_FILE)
print(f"✅ Train loaded: {df_train.shape[0]:,} × {df_train.shape[1]}")
print(f"   Columns: {df_train.columns}")

In [ ]:
# ============================================================
# PREPROCESSING: ENCODING & IMPUTATION
# ============================================================

def preprocess_dataframe(df: pl.DataFrame) -> pl.DataFrame:
    """Apply encoding and imputation to dataframe."""
    
    # Gender encoding
    if 'gender' in df.columns and 'gender_encoded' not in df.columns:
        dtype = str(df['gender'].dtype)
        if 'String' in dtype or 'Utf8' in dtype:
            df = df.with_columns(
                pl.col('gender').map_elements(
                    lambda x: GENDER_MAP.get(x, -1) if x else -1, 
                    return_dtype=pl.Int32
                ).alias('gender_encoded')
            )
        else:
            df = df.with_columns(
                pl.col('gender').fill_null(-1).cast(pl.Int32).alias('gender_encoded')
            )
    elif 'gender_encoded' not in df.columns:
        df = df.with_columns(pl.lit(-1).alias('gender_encoded'))
    
    # Insulin type encoding
    if 'insulin_type_bolus' in df.columns and 'insulin_type_encoded' not in df.columns:
        dtype = str(df['insulin_type_bolus'].dtype)
        if 'String' in dtype or 'Utf8' in dtype:
            df = df.with_columns(
                pl.col('insulin_type_bolus').map_elements(
                    lambda x: INSULIN_MAP.get(x, -1) if x else -1, 
                    return_dtype=pl.Int32
                ).alias('insulin_type_encoded')
            )
        else:
            df = df.with_columns(
                pl.col('insulin_type_bolus').fill_null(-1).cast(pl.Int32).alias('insulin_type_encoded')
            )
    elif 'insulin_type_encoded' not in df.columns:
        df = df.with_columns(pl.lit(-1).alias('insulin_type_encoded'))
    
    # Impute missing numeric columns
    for col in ['weight', 'basal', 'age', 'is_weekend']:
        if col in df.columns:
            df = df.with_columns(pl.col(col).fill_null(-1.0).cast(pl.Float32))
        else:
            df = df.with_columns(pl.lit(-1.0).cast(pl.Float32).alias(col))
    
    return df

# Apply preprocessing
df_train = preprocess_dataframe(df_train)
print("✅ Preprocessing complete")

In [ ]:
# ============================================================
# CALCULATE IMBALANCE RATIO (CRITICAL!)
# ============================================================

# Get available features
available_features = [f for f in FEATURES if f in df_train.columns]
missing_features = [f for f in FEATURES if f not in df_train.columns]

if missing_features:
    print(f"⚠️ Missing features (filled with 0): {missing_features}")
    for f in missing_features:
        df_train = df_train.with_columns(pl.lit(0.0).alias(f))
    available_features = FEATURES

# Prepare arrays
X_train = df_train.select(available_features).to_numpy().astype(np.float32)
y_train = df_train.select(TARGET).to_numpy().ravel().astype(np.float32)

# Handle NaN/Inf
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
y_train = np.nan_to_num(y_train, nan=0.0, posinf=0.0, neginf=0.0)

# Calculate imbalance ratio
num_zeros = (y_train == 0).sum()
num_positive = (y_train > 0).sum()
imbalance_ratio = float(num_zeros / num_positive) if num_positive > 0 else 1.0

print("\n" + "=" * 60)
print("🧮 IMBALANCE ANALYSIS")
print("=" * 60)
print(f"   Total samples:  {len(y_train):,}")
print(f"   Zero values:    {num_zeros:,} ({num_zeros/len(y_train)*100:.1f}%)")
print(f"   Positive values: {num_positive:,} ({num_positive/len(y_train)*100:.1f}%)")
print(f"\n   📊 Calculated Ratio: {imbalance_ratio:.2f}")
print(f"   ⚠️ Previous config used: 50 (WRONG!)")

# Memory cleanup
del df_train
gc.collect()

## ⚗️ Step 2: Define Stabilized Parameters

In [ ]:
# ============================================================
# STABILIZED XGBOOST PARAMETERS
# ============================================================

# Calculate scale_pos_weight with safety margin
SCALE_POS_WEIGHT = imbalance_ratio * 1.5  # 1.5x multiplier for safety

PARAMS = {
    # Objective
    'objective': 'reg:squarederror',
    
    # GPU Acceleration
    'tree_method': 'hist',
    'device': 'cuda',
    
    # ⚡ BALANCING MAGIC (Zero-Inflated Regression)
    'scale_pos_weight': SCALE_POS_WEIGHT,  # Dynamic, not hardcoded!
    'max_delta_step': 1,                    # CRITICAL: Prevents exploding gradients
    'min_child_weight': 5,                  # Requires evidence for bolus prediction
    
    # Hyperparameters
    'learning_rate': 0.02,     # Slower for precision
    'max_depth': 9,            # Deep enough for patterns
    'subsample': 0.7,
    'colsample_bytree': 0.8,
    
    # Regularization
    'gamma': 0.2,              # Minimum loss for split
    'reg_alpha': 1,            # L1
    'reg_lambda': 1,           # L2
    
    # Other
    'seed': 42,
    'verbosity': 1
}

NUM_ROUNDS = 1500
EARLY_STOPPING = 50

print("\n" + "=" * 60)
print("⚗️ STABILIZED PARAMETERS")
print("=" * 60)
print(f"   scale_pos_weight: {SCALE_POS_WEIGHT:.2f} (was: 50)")
print(f"   max_delta_step:   {PARAMS['max_delta_step']} (was: missing!)")
print(f"   min_child_weight: {PARAMS['min_child_weight']}")
print(f"   learning_rate:    {PARAMS['learning_rate']}")
print(f"   max_depth:        {PARAMS['max_depth']}")
print(f"   num_rounds:       {NUM_ROUNDS}")
print(f"   early_stopping:   {EARLY_STOPPING}")

## 📊 Step 3: Load Validation Data

In [ ]:
# ============================================================
# LOAD VALIDATION DATA
# ============================================================
print("\n" + "=" * 60)
print("📊 LOADING VALIDATION DATA")
print("=" * 60)

df_val = pl.read_parquet(VAL_FILE)
print(f"✅ Val loaded: {df_val.shape[0]:,} × {df_val.shape[1]}")

# Preprocess validation data
df_val = preprocess_dataframe(df_val)

# Fill missing features
for f in available_features:
    if f not in df_val.columns:
        df_val = df_val.with_columns(pl.lit(0.0).alias(f))

# Prepare arrays
X_val = df_val.select(available_features).to_numpy().astype(np.float32)
y_val = df_val.select(TARGET).to_numpy().ravel().astype(np.float32) if TARGET in df_val.columns else None

# Handle NaN
X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
if y_val is not None:
    valid_mask = ~np.isnan(y_val)
    X_val = X_val[valid_mask]
    y_val = y_val[valid_mask]
    print(f"✅ Val prepared: {len(y_val):,} valid samples")

del df_val
gc.collect()

## 🚀 Step 4: Training with Early Stopping

In [ ]:
# ============================================================
# CREATE DMATRIX
# ============================================================
print("\n" + "=" * 60)
print("🚀 TRAINING (GPU + Early Stopping)")
print("=" * 60)

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=available_features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=available_features)

evals = [(dtrain, 'train'), (dval, 'val')]

print(f"   Train DMatrix: {X_train.shape}")
print(f"   Val DMatrix:   {X_val.shape}")

In [ ]:
# ============================================================
# TRAIN MODEL
# ============================================================

print(f"\n🏋️ Starting training ({NUM_ROUNDS} rounds, early_stop={EARLY_STOPPING})...")
t0 = time.time()

evals_result = {}

model = xgb.train(
    PARAMS,
    dtrain,
    num_boost_round=NUM_ROUNDS,
    evals=evals,
    early_stopping_rounds=EARLY_STOPPING,
    evals_result=evals_result,
    verbose_eval=100  # Print every 100 rounds
)

train_time = time.time() - t0
print(f"\n✅ Training complete in {train_time:.1f}s")
print(f"   Best iteration: {model.best_iteration}")

## 📈 Step 5: Evaluation & Sanity Check

In [ ]:
# ============================================================
# VALIDATION METRICS
# ============================================================
print("\n" + "=" * 60)
print("📈 VALIDATION METRICS")
print("=" * 60)

y_pred = model.predict(dval)
y_pred = np.maximum(y_pred, 0)  # Clip negatives

# Global metrics
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)

print(f"\n   Global RMSE: {rmse:.4f}")
print(f"   Global MAE:  {mae:.4f}")

# Positive-only metrics
pos_mask = y_val > 0
if pos_mask.sum() > 0:
    pos_mae = mean_absolute_error(y_val[pos_mask], y_pred[pos_mask])
    print(f"   Positive MAE: {pos_mae:.4f} (n={pos_mask.sum():,})")

# Miss rate
DOSE_THRESHOLD = 0.5
missed = ((y_val > DOSE_THRESHOLD) & (y_pred < DOSE_THRESHOLD)).sum()
miss_rate = missed / pos_mask.sum() * 100 if pos_mask.sum() > 0 else 0
print(f"\n   ⚠️ Miss Rate: {miss_rate:.2f}% ({missed:,} missed doses)")

In [ ]:
# ============================================================
# SANITY CHECK: HIGH-CARB PREDICTION
# ============================================================
print("\n" + "=" * 60)
print("🧪 SANITY CHECK: High-Carb Scenario")
print("=" * 60)

# Create dummy high-carb input
sanity_input = {
    'CGM': 180.0,              # High glucose
    'carbs': 100.0,            # High carbs (big meal)
    'time_sin': 0.5,
    'time_cos': 0.866,
    'is_weekend': 0.0,
    'cgm_delta_5min': 10.0,    # Rising glucose
    'cgm_delta_15min': 25.0,
    'basal': 1.0,
    'age': 35.0,
    'weight': 75.0,
    'gender_encoded': 0.0,
    'insulin_type_encoded': 3.0
}

# Create array
sanity_X = np.array([[sanity_input[f] for f in available_features]], dtype=np.float32)
sanity_dmatrix = xgb.DMatrix(sanity_X, feature_names=available_features)

# Predict
sanity_pred = model.predict(sanity_dmatrix)[0]

print(f"\n   Input: CGM={sanity_input['CGM']}, Carbs={sanity_input['carbs']}g")
print(f"   Predicted Bolus: {sanity_pred:.2f} units")

if sanity_pred > 0:
    print(f"   ✅ PASS: Model predicts positive dose for high-carb input")
else:
    print(f"   ❌ FAIL: Model predicts zero for high-carb input!")

In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================
print("\n" + "=" * 60)
print("📊 FEATURE IMPORTANCE (Gain)")
print("=" * 60)

importance = model.get_score(importance_type='gain')
sorted_imp = sorted(importance.items(), key=lambda x: x[1], reverse=True)

for feat, gain in sorted_imp:
    bar = '█' * int(gain / max(importance.values()) * 40)
    print(f"   {feat:25s} {gain:10.2f} {bar}")

## 💾 Step 6: Save Model

In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================
print("\n" + "=" * 60)
print("💾 SAVING MODEL")
print("=" * 60)

model.save_model(MODEL_OUTPUT)
print(f"✅ Model saved: {MODEL_OUTPUT}")

# Verify file size
file_size = os.path.getsize(MODEL_OUTPUT) / (1024*1024)
print(f"   Size: {file_size:.2f} MB")

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("🎉 RETRAINING COMPLETE")
print("=" * 70)

print(f"\n📊 Configuration:")
print(f"   scale_pos_weight: {SCALE_POS_WEIGHT:.2f}")
print(f"   max_delta_step:   {PARAMS['max_delta_step']}")
print(f"   Best iteration:   {model.best_iteration}")

print(f"\n📈 Performance:")
print(f"   Global RMSE:  {rmse:.4f}")
print(f"   Positive MAE: {pos_mae:.4f}")
print(f"   Miss Rate:    {miss_rate:.2f}%")

print(f"\n🧪 Sanity Check:")
print(f"   High-carb prediction: {sanity_pred:.2f} units")

print(f"\n💾 Output:")
print(f"   {MODEL_OUTPUT}")

print("\n" + "=" * 70)